# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient
from lib.tooling import tool

In [3]:
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
tavily = TavilyClient(api_key=TAVILY_API_KEY)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_or_create_collection(name="udaplay4")

print(f"API Key loaded: {OPENAI_API_KEY[:15]}...")
print("Setup complete!")

API Key loaded: voc-78611856416...
Setup complete!


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds most results in the vector DB.
    Args:
        query: a question about game industry.
    Returns a list of games with Platform, Name, YearOfRelease, and Description.
    """
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    
    if not results['documents'][0]:
        return "No games found in the database."
    
    output = []
    for i, doc in enumerate(results['documents'][0]):
        metadata = results['metadatas'][0][i]
        output.append(f"Game {i+1}:\n"
                      f"  Name: {metadata.get('Name', 'N/A')}\n"
                      f"  Platform: {metadata.get('Platform', 'N/A')}\n"
                      f"  Year: {metadata.get('YearOfRelease', 'N/A')}\n"
                      f"  Publisher: {metadata.get('Publisher', 'N/A')}\n"
                      f"  Description: {metadata.get('Description', 'N/A')}")
    
    return "\n\n".join(output)

#### Evaluate Retrieval Tool

In [5]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    Args:
        question: original question from user
        retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    Returns useful: whether the documents are useful, and description of the evaluation result.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an evaluator. Assess if the provided documents can answer the question. Respond with JSON: {\"useful\": true/false, \"description\": \"explanation\"}"},
            {"role": "user", "content": f"Question: {question}\n\nDocuments:\n{retrieved_docs}"}
        ]
    )
    
    result = response.choices[0].message.content
    try:
        parsed = json.loads(result)
        return f"Useful: {parsed.get('useful', False)}\nDescription: {parsed.get('description', result)}"
    except:
        return f"Useful: False\nDescription: {result}"

#### Game Web Search Tool

In [6]:
@tool
def game_web_search(question: str) -> str:
    """
    Semantic search: Finds most results in the vector DB.
    Args:
        question: a question about game industry.
    Returns search results from the web about video games.
    """
    results = tavily.search(query=question, max_results=3)
    
    output = []
    for r in results.get('results', []):
        output.append(f"Title: {r.get('title', 'N/A')}\n"
                     f"URL: {r.get('url', 'N/A')}\n"
                     f"Content: {r.get('content', 'N/A')}")
    
    return "\n\n".join(output) if output else "No web results found."

### Agent

In [7]:
class SimpleAgent:
    def __init__(self, instructions, tools, model="gpt-4o-mini"):
        self.instructions = instructions
        self.tools = {t.name: t for t in tools}
        self.model = model
        self.tool_schemas = [t.dict() for t in tools]
        self.messages = []

    def invoke(self, query):
        self.messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": query}
        ]
        
        while True:
            response = client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=self.tool_schemas,
                tool_choice="auto"
            )
            
            msg = response.choices[0].message
            
            if msg.tool_calls:
                self.messages.append(msg)
                for tc in msg.tool_calls:
                    fn_name = tc.function.name
                    fn_args = json.loads(tc.function.arguments)
                    result = str(self.tools[fn_name](**fn_args))
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result
                    })
            else:
                return msg.content

agent = SimpleAgent(
    instructions="""You are UdaPlay, an AI Research Agent for the video game industry.
1. First use retrieve_game to search the internal database
2. Use evaluate_retrieval to assess if the results are useful
3. If not useful, use game_web_search to find the answer online
4. Provide a clear, structured answer with citations.""",
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)

print("Agent created!")

Agent created!


In [10]:
import requests

response = requests.post(
    "https://openai.vocareum.com/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": "Say hello"}]
    }
)
print(response.status_code)
print(response.text[:500])

400
{"error":{"code":null,"message":"This key was not found. Please check key was inputed correctly.","param":null,"type":"invalid_request_error"}}


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes